# DeepCardio-RAG — Genuine Training on Benchmark Datasets (two runtimes)

Trains every **real** module on standard benchmark datasets and reports **held-out** metrics only.
Directly answers the RR review: no random-weight inference, no literature/benchmark values copied
as results, no train-set evaluation.

---

## Which cells run where

Section 0 auto-detects the runtime and prints which one you are on. **Re-run section 0 and the
dependency cell after every runtime switch** — nothing carries over between backends.

| Section | Cells | Runtime | Why |
|---|---|---|---|
| 0–2 setup | 1–4 | **either** | runtime detect + deps |
| 3 Kaggle creds | 5–6 | **Colab T4** | dataset downloads |
| 3b–3c datasets | 7–10 | **Colab T4** | PTB-XL / VFDB / CirCor fetch |
| 4 train all | 11–12 | **Colab T4** | GPU training |
| 5 individual modules | 13–16, 18 | **Colab T4** | GPU training |
| 5 Milvus client | 17 | **local** | pinned `pymilvus==2.6.9` |
| 5b new modules | 19–23 | **Colab T4** | GPU training |
| 5c Milvus check | 24–26 | **local** | needs `localhost:19530` |
| 6 confirm pipeline | 27–28 | **local** | exercises retrieval |
| 7 RAG eval | 29–30 | **either** | in-memory, no vector DB |
| 8 ablations | 31–32 | **Colab T4** | GPU training |

**Why the split works without a tunnel:** no training module touches the vector DB
(`grep db_manager|get_collection|init_db|pymilvus` over `core/train_*.py` and
`colab_train_genuine.py` returns nothing). The two halves never talk during a run, so a Colab
cloud VM never needs to reach your private machine. The handoff is Drive itself — weights written
to `data/` on Colab sync back down to `G:\My Drive\DeepCardio-RAG\data\`.

**Order:** run the Colab T4 cells → wait for Drive to finish syncing (`ecg_ptbxl_encoder.pt`
is ~55 MB) → *Connect ▾ → Connect to a local runtime* → re-run section 0 → run the local cells.

> Colab: `Runtime → Change runtime type → T4 GPU`.
> Local runtime prerequisites on the Milvus box:
> `pip install jupyter_http_over_ws notebook` →
> `jupyter server extension enable --py jupyter_http_over_ws` →
> `jupyter notebook --NotebookApp.allow_origin="https://colab.research.google.com" --port=8888 --NotebookApp.port_retries=0`

---

| Module | Benchmark dataset | Split | Reported metrics |
|---|---|---|---|
| Flagship ECG encoder | **PTB-XL** 12-lead (PhysioNet) | patient-independent folds 1-8/9/10 | acc, macro-F1, per-class AUC, Brier, ECE |
| ECG Arrhythmia CNN | **MIT-BIH** beats | official train/test | acc, macro-F1, AUC (OvR) |
| Arthritis ensemble | **NHANES / BRFSS** (Kaggle) | repeated Stratified K-Fold | acc/AUC/F1 mean ± 95% CI + Brier |
| CardioFusion (hybrid) | **MIT-BIH** (ECG-signal path) | train / **val** (selection) / held-out test | macro-F1, AUC (honest headline), acc; **learned** task weights |

**Honesty note on CardioFusion:** its *joint 4-modality fusion + contrastive alignment* needs same-patient paired data across all modalities, which no public benchmark provides. So we train its ECG-signal path genuinely on MIT-BIH and replace the hand-set task weights with **learned homoscedastic uncertainty weighting** (Kendall et al., CVPR 2018) — the citable answer to *"how is the CardioFusion weighting determined?"*

Model selection uses a **validation split carved from train** — the held-out test set is never used to choose a checkpoint (this removes the test-set selection bias of the earlier run). The arrhythmia loss is **class-weighted** (inverse frequency) to counter MIT-BIH's heavy Normal-class imbalance, so the model no longer collapses to the majority class. Consequently *macro-F1 / AUC* are the honest headline; test **accuracy** is deliberately traded down for minority-class recall and should **not** be read as the main result.

## 1. Mount Drive & enter the project

In [ ]:
# [BOTH] Runtime detect. Re-run this cell after EVERY runtime switch — nothing carries over.
# Colab cloud -> CODE from GitHub, data + weights from Drive.
# Local runtime -> uses the local repo and reaches Milvus.
import os, sys, shutil, subprocess

try:
    from google.colab import drive          # exists ONLY on a Colab-hosted runtime
    IN_COLAB_CLOUD = True
except ModuleNotFoundError:
    IN_COLAB_CLOUD = False

if IN_COLAB_CLOUD:
    drive.mount('/content/drive')

    # Code comes from GitHub so teammates' pushes reach every training run.
    # Never run git against /content/drive/... — Drive syncs .git while git is
    # writing to it, which corrupts the repo. The Drive copy is DATA ONLY.
    CODE       = '/content/DeepCardio-RAG'
    DRIVE_DATA = '/content/drive/MyDrive/DeepCardio-RAG/data'
    PUBLIC_URL = 'https://github.com/Rama2105/DeepCardio-RAG.git'

    if os.path.isdir(os.path.join(CODE, '.git')):
        subprocess.run(['git', '-C', CODE, 'pull', '--ff-only'], check=False)
    else:
        try:
            from google.colab import userdata          # Colab Secrets (key icon)
            token = userdata.get('GITHUB_TOKEN')
        except Exception:
            from getpass import getpass
            token = getpass('GitHub token: ')
        # capture_output so the token in the URL never reaches the saved cell output
        subprocess.run(
            ['git', 'clone',
             f'https://{token}@github.com/Rama2105/DeepCardio-RAG.git', CODE],
            check=True, capture_output=True)
        subprocess.run(['git', '-C', CODE, 'remote', 'set-url', 'origin', PUBLIC_URL],
                       check=True)

    # Datasets and trained weights live on Drive and persist across runtimes; the
    # repo excludes them. Symlinking keeps every relative path valid unchanged
    # (data/vfdb/, data/circor-heart-sound/, data/*.pt).
    link = os.path.join(CODE, 'data')
    if os.path.islink(link):
        os.unlink(link)
    elif os.path.isdir(link):
        shutil.rmtree(link)
    os.makedirs(DRIVE_DATA, exist_ok=True)
    os.symlink(DRIVE_DATA, link)

    os.chdir(CODE)
    rev = subprocess.run(['git', 'log', '--oneline', '-1'],
                         capture_output=True, text=True).stdout.strip()
    print('code rev:', rev)
else:
    # ---- LOCAL RUNTIME (this machine hosts the Milvus standalone server) ----
    candidates = [
        os.environ.get('DEEPCARDIO_DIR'),
        r'G:\My Drive\DeepCardio-RAG',              # Windows box (Google Drive folder)
        os.path.expanduser('~/DeepCardio-RAG'),     # legacy Linux box (10.228.1.9)
    ]
    repo = next((p for p in candidates if p and os.path.isdir(p)), None)
    if repo is None:
        raise SystemExit(
            "Repo not found. Set DEEPCARDIO_DIR to the project folder first, e.g.\n"
            "    os.environ['DEEPCARDIO_DIR'] = r'G:\\My Drive\\DeepCardio-RAG'"
        )
    os.chdir(repo)
    # Set BOTH host and port. .env can still hold stale tunnel values (bore.pub:38467)
    # and pydantic gives real environment variables precedence over .env, so setting
    # only the host leaves the port aimed at a dead tunnel.
    os.environ.setdefault('MILVUS_HOST', 'localhost')
    os.environ.setdefault('MILVUS_PORT', '19530')

# Without this, `import core.pipeline` raises ModuleNotFoundError whenever the kernel's
# working directory is not the repo root — the failure seen in section 6 previously.
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# GPU check in Python, not `!nvidia-smi`: that is bash and fails on a Windows local
# runtime, where the ! shell is cmd.exe.
try:
    import torch
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none — CPU mode'
except ModuleNotFoundError:
    gpu = 'torch not installed yet — run the dependencies cell'

print('RUNTIME :', 'Colab cloud' if IN_COLAB_CLOUD else 'LOCAL (Milvus host)')
print('cwd     :', os.getcwd())
print('GPU     :', gpu)
print('MILVUS  :', (os.environ['MILVUS_HOST'] + ':' + os.environ['MILVUS_PORT'])
      if not IN_COLAB_CLOUD else 'not reachable from Colab cloud — see section 5c')


## 2. Install dependencies

In [ ]:
# NOTE: do NOT upgrade torch on Colab - the preinstalled CUDA build works; upgrading it
# breaks transformers' model imports (e.g. 'Could not import GPT2LMHeadModel').
# No blanket --upgrade: it dragged pandas to 3.0.x (breaks the pandas-heavy arthritis
# pipeline + Colab). Pin every version-sensitive package; install the rest.
#
# wfdb>=4.3.1 is REQUIRED, not cosmetic. Older wfdb's rdann() raises
#   OverflowError: Python integer 256 out of bounds for uint8
# under NumPy 2, so VFDB annotations silently fell back to a broken byte parser and
# produced a SINGLE-CLASS dataset that scored accuracy 1.0 (2026-07-30).
# wfdb 4.3.1 in turn requires pandas>=2.2.3, so pandas is pinned to 2.2.3 - NOT
# Colab's 2.2.2 baseline. The resulting "google-colab 1.0.0 requires pandas==2.2.2"
# warning is harmless: that is Colab's own UI tooling, not anything training touches.
!pip -q install "wfdb>=4.3.1" kagglehub scikit-learn "transformers==4.44.2" accelerate "pandas==2.2.3" opencv-python-headless
import torch; print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
import pandas, wfdb, sys
print('pandas', pandas.__version__, '| wfdb', wfdb.__version__)
print('numpy', __import__('numpy').__version__)

pandas 2.2.3 | wfdb 4.3.1
numpy 2.0.2


## 3. Kaggle credentials (for PTB-XL + NHANES/BRFSS downloads)
Get an API token from kaggle.com → Account → *Create New API Token* (downloads `kaggle.json`).
MIT-BIH CSVs already live in `data/` on your Drive, so the arrhythmia + CardioFusion modules need no download.

In [ ]:
import os
os.environ['KAGGLE_USERNAME'] = 'resarch'   # <-- fill in
os.environ['KAGGLE_KEY']      = 'af058634d79a5b99046318212905e5d0'   # <-- fill in
assert os.environ['KAGGLE_USERNAME'] and os.environ['KAGGLE_KEY'], 'Set your Kaggle creds above.'

### 3b. PTB-XL dataset completeness fix (run before training)
Google-Drive-cached copies of PTB-XL are sometimes missing waveform folders (e.g. `records500/21000/`). Missing records silently fall back to **synthetic** signals, which contaminates the results with fake data. This cell force-fetches a **complete** copy to local Colab disk and points the loader at it.

After it runs you should see `records500/21000 .hea files:` ≈ **838** (not 0), and **no** `synthetic fallback` messages during PTB-XL training. If it prints **0**, the Kaggle mirror itself is incomplete — switch the loader to 100 Hz records instead.

In [ ]:
import os, glob, kagglehub
import core.ptbxl_loader as L

os.environ['KAGGLEHUB_CACHE'] = '/content/kaggle_cache'          # local disk, not Drive
path = kagglehub.dataset_download("khyeh0719/ptb-xl-dataset", force_download=True)

root = next(r for r, d, f in os.walk(path) if 'ptbxl_database.csv' in f)
n = len(glob.glob(root + '/records500/21000/*.hea'))
print("root:", root)
print("records500/21000 .hea files:", n)      # want ~800, NOT 0

L.LOCAL_DATASET_DIR = root      # loader now uses this complete copy first
print("Loader pointed at complete local copy ✓")

### 3c. PhysioNet datasets for the newly-trained modules (VFDB + CirCor)

Run this **before** step 4 — `run_all()` trains echo/PCG/VFDB as well, and each raises a clear
error if its dataset is absent. (Every step is guarded, so a missing dataset never aborts the run;
it just records an `error` entry and the other modules continue.)

Both downloads are free and need no PhysioNet login. VFDB is ~100 MB; **CirCor is ~3 GB** and will
dominate the runtime of this cell.

`EchoNet-Dynamic` **cannot** be fetched this way — it requires registration with Stanford AIMI.
If you don't have access yet, just leave it: the echo module will report an error and everything
else still runs.

In [ ]:
# Expected final layout (the train scripts raise a clear error if these are missing):
#   data/vfdb/                              .dat / .hea / .atr
#   data/circor-heart-sound/training_data/  .wav + .tsv
#   data/EchoNet-Dynamic/                   Videos/ + FileList.csv   <- manual (Stanford AIMI)
#
# Datasets persist on Drive, so re-running this is cheap: wget skips what is current.
#
# -N -c is deliberate. -N re-fetches anything whose size/timestamp differs from the
# server; -c resumes a partial file. Do NOT substitute -nc ("no-clobber"): it skips any
# file that merely EXISTS without checking completeness, so a download killed mid-write
# leaves a TRUNCATED file that -nc will never repair. That happened on 2026-07-30 -
# 85286_AV.wav held 15,974 of 161,280 bytes and silently degraded PCG training.

%cd /content/drive/MyDrive/DeepCardio-RAG/data
!wget -r -N -c -np -nH --cut-dirs=3 -P vfdb https://physionet.org/files/vfdb/1.0.0/
!wget -r -N -c -np -nH --cut-dirs=3 -P circor-heart-sound https://physionet.org/files/circor-heart-sound/1.0.3/
%cd /content/drive/MyDrive/DeepCardio-RAG

# PhysioNet ships training_data.csv BESIDE training_data/, but core/train_circor.py
# looks for it INSIDE. Without this copy the PCG module dies with "CirCor not found".
!cp -n data/circor-heart-sound/training_data.csv data/circor-heart-sound/training_data/ 2>/dev/null; true

# Verify before training - anything marked MISS will be skipped with an error in step 4.
import os, glob
for p in ['data/vfdb',
          'data/circor-heart-sound/training_data',
          'data/EchoNet-Dynamic']:
    n = len(os.listdir(p)) if os.path.isdir(p) else 0
    print(f"{'OK  ' if n else 'MISS'}  {p}  ({n} entries)")

# CirCor completeness against PhysioNet's own manifest (RECORDS = 3163 recordings).
_c = 'data/circor-heart-sound'
if os.path.isfile(f'{_c}/RECORDS'):
    want = len(open(f'{_c}/RECORDS').read().split())
    have = {e: len(glob.glob(f'{_c}/training_data/*.{e}')) for e in ('hea', 'wav', 'tsv')}
    ok = all(v == want for v in have.values())
    print(f"CirCor RECORDS={want} | on disk {have} |",
          "OK" if ok else "INCOMPLETE - re-run this cell")

In [ ]:
# [OPTIONAL - takes a few minutes] Byte-level integrity check against PhysioNet's own
# SHA256SUMS.txt. File COUNTS cannot detect truncation: 85286_AV.wav was present and
# counted, but held 15,974 of 161,280 bytes. Run this after any interrupted download.
# Re-fetch whatever it lists with `wget -N` (never -nc).
import hashlib, os
D = 'data/circor-heart-sound'
bad = []
for line in open(f'{D}/SHA256SUMS.txt'):
    want, _, name = line.strip().partition(' ')
    p = os.path.join(D, name.strip())
    if not os.path.isfile(p) or hashlib.sha256(open(p, 'rb').read()).hexdigest() != want:
        bad.append(name.strip())
print('corrupt or missing:', len(bad))
print(bad[:20])

## 4. Train everything (benchmark datasets, held-out evaluation)
`n_records=3000` per dataset for a fast, reproducible run. Use `n_records=None` for the full benchmark.

In [ ]:
from colab_train_genuine import run_all
results = run_all(n_records=3000, ecg_epochs=30, arr_epochs=25, fusion_epochs=20)

## 5. Individual modules (optional — run any one on its own)

In [ ]:
# Flagship ECG encoder on real PTB-XL (saves data/ecg_ptbxl_encoder.pt used by core/pipeline.py)
from core.train_ptbxl import train_ptbxl
ecg = train_ptbxl(n_records=3000, epochs=30)   # patient-independent split, genuine held-out metrics

In [ ]:
# Arthritis ensemble on real Kaggle data with honest repeated-CV (95% CIs)
# Also prints the leakage-free feature importances used for the paper's tab:arfeat.
import importlib, colab_train_genuine as C; importlib.reload(C)   # pick up updated train_arthritis
import json
arth = C.train_arthritis(n_records=3000)
print("cv_auc:", arth['cv_auc_roc'], "| cv_acc:", arth['cv_accuracy'])
print("total_features:", arth['total_features'])
print(json.dumps(arth['top_features'], indent=2))
assert not any(f['feature'] == 'ArthritisType' for f in arth['top_features']), "LEAK: ArthritisType still present"
print("leakage-free OK (ArthritisType absent)")

In [ ]:
# ECG Arrhythmia 1D-CNN on MIT-BIH (data already on Drive)
from colab_train_genuine import train_ecg_arrhythmia
arr = train_ecg_arrhythmia(n_records=3000, epochs=25); arr

In [ ]:
# [LOCAL RUNTIME] Milvus client — PIN to 2.x.
#
# This cell previously read `pip install pymilvus` (unpinned) and its saved output shows
# what that does: it installed pymilvus 3.0.0, whose API drops the 2.x ORM calls
# (connections / Collection / FieldSchema) that database/db_manager.py is built on, AND
# pulled setuptools 83 which violates torch's setuptools<82 requirement.
#
# It also does not belong in the GPU training run: nothing in sections 4-5b touches the
# vector DB. Run it on the LOCAL runtime, before section 5c.
!pip -q install "pymilvus==2.6.9"

import importlib, pymilvus
importlib.reload(pymilvus)
print('pymilvus', pymilvus.__version__, '(want 2.6.9)')
# Still showing 3.x? Restart the kernel and re-run — an already-imported pymilvus
# stays resident in the process until restart.

In [ ]:
# CardioFusion hybrid — genuine ECG-signal path on MIT-BIH, LEARNED task weights
# (saves data/cardiofusion_weights.pt, loaded by get_cardiofusion_model())
# Model selection is on a VAL split carved from train (test evaluated once, at the end);
# the arrhythmia loss is class-weighted for MIT-BIH imbalance. Expect test ACCURACY below
# a majority-class baseline — macro-F1 / AUC are the honest headline, not accuracy.
from core.train_cardiofusion import train_cardiofusion
cf = train_cardiofusion(n_records=3000, epochs=20)
print('learned task weights:', cf['learned_task_weights'])

### 5b. Newly-trained modules (peer-review fixes)
These four modules were flagged by the review as never independently trained (echo/PCG/heart-disease) or running on **random weights** (VFDB — the code-blue safety failure). Each trains on real data with a held-out split and saves weights the pipeline loads.

**Datasets that need manual placement** (PhysioNet access / Stanford AIMI):
- EchoNet-Dynamic -> `data/EchoNet-Dynamic/` (Videos/ + FileList.csv)
- CirCor DigiScope -> `data/circor-heart-sound/training_data/`
- VFDB -> `data/vfdb/` (.dat/.hea/.atr)
- heart.csv (UCI Cleveland) is already in `data/`.


In [ ]:
# Echo LVEF (EchoNet-Dynamic 3D-CNN) — replaces the untrained module that reported
# a physically impossible EF = -0.1%. Predicted EF is clamped to [0,100]% (gating).
from core.train_echonet import train_echonet
echo = train_echonet(n_train=3000, n_eval=1000, epochs=25)   # held-out MAE/R2/cat-AUC


In [ ]:
# VFDB ventricular arrhythmia — GENUINE weights replace the random-init network that
# issued 'code blue'/'defibrillation'. Patient-independent split by recording.
from core.train_vfdb import train_vfdb
vfdb = train_vfdb(epochs=30)   # binary Dangerous-vs-Normal F1/AUC + 4-class rhythm F1


In [ ]:
# PCG murmur (CirCor DigiScope) — real phonocardiogram training, subject-level split.
from core.train_circor import train_circor
pcg = train_circor(n_max=3000, epochs=25)   # 3-class murmur + Present-vs-Absent AUC


In [ ]:
# Heart disease (UCI Cleveland) — DEDUPLICATED 1025->~302 unique rows (M5), leakage-free
# repeated CV, with logistic + GBT baselines (R8): a BERT-MoE must beat plain baselines.
from core.train_heartdisease import train_heartdisease
hd = train_heartdisease(n_repeats=5, n_splits=3, epochs=80)


## 5c. Milvus check — `[LOCAL RUNTIME]`

Milvus standalone runs on the **same machine as this kernel**, reachable at `localhost:19530`.
Run the pinned-client cell in section 5 (`pymilvus==2.6.9`) before these.

**Skip this entire section on a Colab cloud backend.** A cloud VM cannot route to a private
machine, and `db_manager` will silently fall back to FAISS while still logging
`✓ Vector DB ready` — a green message that means nothing. That silent fallback is why earlier
runs served `[DEMO]` guidelines while appearing healthy.

In [ ]:
# 5c.1 — Connect, and check the server matches what config.py expects.  [LOCAL RUNTIME]
# A dim/metric mismatch does NOT fail here; it fails later at RAG insert/search time,
# which is much harder to diagnose. Catch it now.
import os
from pymilvus import connections, utility, Collection
from config import settings

HOST = os.environ.get('MILVUS_HOST', 'localhost')
PORT = os.environ.get('MILVUS_PORT', '19530')

connections.connect(alias='dc', host=HOST, port=PORT, timeout=10)
print(f'✓ connected to {HOST}:{PORT}')
print('  server version :', utility.get_server_version(using='dc'))
print('  collections    :', utility.list_collections(using='dc'))

want = dict(name=settings.collection_name, dim=settings.embedding_dim,
            metric=settings.milvus_metric, index=settings.milvus_index_type)
print('\nconfig expects :', want)

if not utility.has_collection(want['name'], using='dc'):
    print(f"ℹ collection '{want['name']}' does not exist yet — create it with:")
    print("    python -m database.seed_data --force")
else:
    col = Collection(want['name'], using='dc')
    # Find the vector field by asking which field carries a 'dim' param. Do NOT filter on
    # str(field.dtype).endswith('FLOAT_VECTOR') — that silently matches nothing on
    # pymilvus 2.6.9, leaving dim=None and raising a false DIM MISMATCH alarm.
    got_dim = next((int(p['dim']) for f in col.schema.fields
                    if (p := (getattr(f, 'params', None) or {})).get('dim')), None)
    print('server has     :', dict(name=want['name'], dim=got_dim, rows=col.num_entities))
    for idx in col.indexes:
        print('  index        :', idx.params)

    if got_dim is None:
        print('⚠ could not read the vector dimension from the schema — inspect col.schema manually')
    elif got_dim != want['dim']:
        print(f"⚠ DIM MISMATCH: server={got_dim} vs config={want['dim']}."
              " Re-seed with --force or every RAG insert will fail.")
    else:
        print('✓ dimension matches config')

In [ ]:
# 5c.2 — The check that actually matters: is the PROJECT on real Milvus?  [LOCAL RUNTIME]
# backend=='milvus' means the real server. Anything else means core/pipeline.py is
# serving hardcoded [DEMO] guidelines instead of retrieved documents — while still
# logging "✓ Vector DB ready". Never trust that log line on its own.
from database.db_manager import reset_db, get_db_status
from database.seed_data import MOCK_KNOWLEDGE_BASE

reset_db()                       # clear any cached instance so this reflects the current env
st = get_db_status()
print('backend   :', st['backend'])
print('status    :', st['status'])
print('host      :', st['host'])
print('documents :', st['documents'])
print('dim/metric:', st['embedding_dim'], '/', st['metric'], '| index', st['index_type'])

expected = len(MOCK_KNOWLEDGE_BASE)
if st['backend'] != 'external Milvus':
    print(f"\n⚠ NOT on real Milvus. Check: pymilvus 2.6.9 installed (section 5)?"
          f" MILVUS_HOST/PORT correct? container up (`docker ps`)?")
elif st['documents'] != expected:
    print(f"\n⚠ Server has {st['documents']} docs but seed_data defines {expected}."
          f" The corpus changed since seeding — re-seed:  python -m database.seed_data --force")
else:
    print(f"\n✓ RAG on real Milvus with the current corpus ({expected} documents)")

# Per-superclass coverage: a class with 0 documents cannot be retrieved for, and the
# pipeline queries by PTB-XL superclass.
cov = {}
for d in MOCK_KNOWLEDGE_BASE:
    cov[d.get('superclass', 'unclassified')] = cov.get(d.get('superclass', 'unclassified'), 0) + 1
print('corpus coverage:', {k: cov[k] for k in ('NORM', 'MI', 'STTC', 'CD', 'HYP') if k in cov})

## 6. Confirm the flagship pipeline uses the TRAINED encoder  `[LOCAL RUNTIME]`
After sections 4/5, `get_model()` loads the genuine PTB-XL weights instead of random init.

Tagged **[LOCAL RUNTIME]** because it now also exercises retrieval, which needs Milvus. On a Colab **cloud** backend the retrieval half reports a fallback no matter how well training went.

In [ ]:
import importlib, torch, core.pipeline as P
importlib.reload(P)
model = P.get_model()

# Encoder and diagnostic head come from the SAME PTB-XL checkpoint but do different
# jobs: the encoder makes the embedding real, the head is what retrieval is queried with.
print('Flagship ECG encoder trained  :', model.encoder_is_trained)
print('PTB-XL diagnostic head loaded :', model.diagnostic_head is not None)
print('Diagnostic classes            :', model.diagnostic_classes or '(none)')

# Retrieval is diagnosis-grounded only when that head is loaded. Without it the pipeline
# deliberately serves [DEMO] placeholders rather than querying Milvus with something
# unfounded — so 'demo-fallback' is honest behaviour, but the guidelines below are then
# NOT about this ECG and must not be shown as retrieved evidence.
with torch.no_grad():
    out = model(torch.randn(1, 12, 5000))

src = out['retrieval_source']            # hybrid-rrf | milvus-sbert | demo-fallback
print('\nRetrieval source              :', src)
print('Retrieval query               :', out['retrieval_queries'][0])
print('Predicted diagnosis           :', out['diagnoses'][0])
print('\nTop retrieved documents:')
for m in out['context_meta'][0][:3]:
    # rrf_score / ranks are present only under hybrid fusion.
    extra = ''
    if m.get('rrf_score') is not None:
        extra = f" (rrf {m['rrf_score']:.5f}, sem#{m.get('semantic_rank')}, lex#{m.get('lexical_rank')})"
    print(f"  [{m.get('score')}]{extra} {str(m.get('text'))[:76]}")

# Test MEMBERSHIP, not equality against one literal — the source name changes when the
# retrieval strategy changes, and an `== 'milvus-sbert'` check silently reports
# correctly-retrieved documents as ungrounded.
if src not in P.GROUNDED_RETRIEVAL_SOURCES:
    print('\n⚠ NOT grounded. Check in order:'
          '\n   1. Is this the LOCAL runtime? Colab cloud cannot reach your Milvus.'
          '\n   2. Did section 5c report backend=external Milvus?'
          '\n   3. Has data/ecg_ptbxl_encoder.pt finished syncing down from Drive?')
else:
    print(f'\n✓ Retrieval is grounded (source={src})')

## 7. Retrieval-quality evaluation (RAG metrics — review point #6)
Standard IR metrics on a labelled clinical benchmark: semantic Sentence-BERT vs a lexical TF-IDF baseline.

In [ ]:
from core.rag_eval import run_rag_evaluation
rag = run_rag_evaluation(include_semantic=True)   # Recall@k, Precision@k, MRR, nDCG@k, Hit@k

## 8. Ablation studies (review point #5)
Each study varies ONE component and reports the change in genuine performance.
**Note:** the arthritis ablation revealed & fixed a target-leakage feature (NHANES `MCQ195`), dropping AUC from an inflated ~0.99 to an honest ~0.79.

In [ ]:
# Ablation 1: Arthritis ensemble vs each component vs logistic baseline (CPU-fast)
from core.ablation import ablate_arthritis, ablate_ecg_encoder
_ = ablate_arthritis(n_records=3000)

# Ablation 2: flagship ECGTransformerMoE vs ResNet-34 CNN baseline on PTB-XL (GPU)
_ = ablate_ecg_encoder(n_records=3000, epochs=30)

### 5d. VFDB recording-level cross-validation  `[Colab T4]`

The single 15/3/4 split reported on 2026-07-30 gave test F1(dang) 0.5147 / AUC 0.8046
against val F1 0.894 / AUC 0.984. That gap over **4 test recordings** means one split
of 22 recordings cannot support a point estimate. This replaces it with a distribution:
every recording is a test recording exactly once per repeat, folds are whole recordings
stratified by dangerous-window fraction, and validation always comes out of the training
pool. Quote the mean and interval, not the single split.

Runtime scales as `n_folds x n_repeats` fits (default 5x2 = 10). Windows are extracted
once and cached, so most of the cost is training, not I/O.

In [ ]:
# VFDB recording-level CV — replaces the fragile single-split point estimate.
# 5 folds x 2 repeats = 10 fits. Drop to n_repeats=1 / epochs=10 for a quick check.
from core.cv_vfdb import cross_validate_vfdb

cv = cross_validate_vfdb(n_folds=5, n_repeats=2, epochs=30)   # -> data/vfdb_cv_metrics.json

# Headline number to quote, with its spread across folds:
auc = cv['aggregate']['binary_auc']
print(f"\nVFDB dangerous-vs-normal AUC: {auc['mean']} "
      f"[{auc['ci95_low']}, {auc['ci95_high']}] over {auc['n_folds_used']} folds")

[VFDB] Loaded 22 records | 592 rhythm annotations | 239 dangerous episodes.
[VFDB-CV] extracting 5s windows from 22 recordings (once; cached across all folds)...
[VFDB-CV] 22 recordings | 9218 windows | dangerous fraction 0.216
[VFDB-CV] per-recording dangerous fraction: min 0.014 / median 0.152 / max 0.792
[VFDB-CV] rep1/fold1 | train 14 rec / 5866 win | val 3 rec | test 5 rec / 2095 win | test dangerous frac 0.155
[VFDB-CV] rep1/fold1 -> AUC 0.9821 | F1(dang) 0.6963 | rhythm macro-F1 0.4481
[VFDB-CV] rep1/fold2 | train 15 rec / 6285 win | val 3 rec | test 4 rec / 1676 win | test dangerous frac 0.342
[VFDB-CV] rep1/fold2 -> AUC 0.9369 | F1(dang) 0.7668 | rhythm macro-F1 0.525
[VFDB-CV] rep1/fold3 | train 14 rec / 5866 win | val 3 rec | test 5 rec / 2095 win | test dangerous frac 0.163
[VFDB-CV] rep1/fold3 -> AUC 0.9453 | F1(dang) 0.7341 | rhythm macro-F1 0.4852
[VFDB-CV] rep1/fold4 | train 15 rec / 6285 win | val 3 rec | test 4 rec / 1676 win | test dangerous frac 0.199
[VFDB-CV] rep1